# Downloading the Dataset

1. Step one is to install the necessary pacakages

In [ ]:
%pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client


2. Step two is importing the needed libraries

In [1]:
import os
import zipfile
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
import io
import pickle

3. Defining the Google Drive folder and local storage paths

In [3]:

DRIVE_FOLDER_NAME = "MGlyphsStars"
LOCAL_FOLDER_PATH = "C:/Users/mohanned/Desktop/Mohaned/Data"  # Change this path to where you want to store the data


4. Authenticate with Google Drive API 

In [4]:
SCOPES = ["https://www.googleapis.com/auth/drive"]
CREDS_PATH = "credentials.json"  

5. Defining necessary functions

In [58]:
def authenticate_drive():
    creds = None
    if os.path.exists("token.pickle"):
        with open("token.pickle", "rb") as token:
            creds = pickle.load(token)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CREDS_PATH, SCOPES)
            creds = flow.run_local_server(port=0)
        with open("token.pickle", "wb") as token:
            pickle.dump(creds, token)
    return build("drive", "v3", credentials=creds)

In [59]:
def get_folder_id(service, folder_name):
    """Get Google Drive folder ID by name"""
    query = f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
    results = service.files().list(q=query, fields="files(id)").execute()
    folders = results.get("files", [])
    return folders[0]["id"] if folders else None

In [60]:
def list_mglyph_files(service, folder_id):
    """List all .mglyph files (ZIP format) in the specified Google Drive folder"""
    query = f"'{folder_id}' in parents and trashed=false and mimeType='application/x-zip'"
    results = service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get("files", [])

    if not files:
        print("No .mglyph files found in Drive folder.")
    else:
        print(f"Found {len(files)} .mglyph files to download.")

    return files


In [61]:
def download_file(service, file_id, file_name, destination):
    """Download a file from Google Drive"""
    request = service.files().get_media(fileId=file_id)
    file_path = os.path.join(destination, file_name)
    with open(file_path, "wb") as f:
        downloader = MediaIoBaseDownload(f, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return file_path

In [66]:
def unzip_and_clean(file_path, extract_to):
    """Extract .mglyph (ZIP) files into individual folders and remove them after extraction"""
    
    # Get the base name (without extension) to use as the folder name
    base_name = os.path.splitext(os.path.basename(file_path))[0]
    extract_path = os.path.join(extract_to, base_name)  # Create a unique folder
    
    # Check if the folder already exists
    if os.path.exists(extract_path):
        print(f"Skipping {base_name}: Folder already exists.")
        return  # Skip extraction

    # Ensure the folder exists
    os.makedirs(extract_path, exist_ok=True)

    temp_zip_path = file_path + ".zip"  # Temporary rename to .zip
    
    # Rename the .mglyph file to .zip for extraction
    os.rename(file_path, temp_zip_path)

    try:
        with zipfile.ZipFile(temp_zip_path, "r") as zip_ref:
            zip_ref.extractall(extract_path)  # Extract into its own folder
        
        os.remove(temp_zip_path)  # Remove the renamed .zip file
        print(f"Extracted and cleaned up: {file_path} -> {extract_path}")

    except zipfile.BadZipFile:
        print(f"Error: {file_path} is not a valid ZIP file.")
        
        # Restore original name if extraction fails
        os.rename(temp_zip_path, file_path)


I added this function to check if the files are there (If you want to use it please make sure to uncomment the debug file listing in main function)

In [67]:
def debug_list_files(service, folder_id):
    """List all files in the specified Google Drive folder (for debugging)"""
    query = f"'{folder_id}' in parents and trashed=false"
    results = service.files().list(q=query, fields="files(id, name, mimeType)").execute()
    files = results.get("files", [])
    
    if not files:
        print("No files found in the folder.")
    else:
        print("Files found in Drive folder:")
        for f in files:
            print(f" - {f['name']} (MIME type: {f['mimeType']})")
    
    return files


In [68]:
def main():
    # Authenticate and get service
    service = authenticate_drive()

    # Get Google Drive folder ID
    folder_id = get_folder_id(service, DRIVE_FOLDER_NAME)
    if not folder_id:
        print(f"Folder '{DRIVE_FOLDER_NAME}' not found in Drive.")
        return
    # Debug file listing
    #all_files = debug_list_files(service, folder_id) ##You can uncomment this to check all the files within the drive folder!!!!!!!

    # List zipped files in Drive folder
    zipped_files = list_mglyph_files(service, folder_id)
    if not zipped_files:
        print("No zip files found in Drive folder.")
        return

    print(f"Found {len(zipped_files)} zip files in Drive folder.")

    # Ensure the local folder exists
    os.makedirs(LOCAL_FOLDER_PATH, exist_ok=True)

    # Check already extracted folders
    extracted_folders = set(os.listdir(LOCAL_FOLDER_PATH))

    for file in zipped_files:
        zip_name = file["name"]
        zip_path = os.path.join(LOCAL_FOLDER_PATH, zip_name)
        extracted_folder_name = zip_name.replace(".zip", "")
        
        # Skip if already extracted
        if extracted_folder_name in extracted_folders:
            print(f"Skipping '{zip_name}', already extracted.")
            continue

        print(f"Downloading {zip_name}...")
        downloaded_zip_path = download_file(service, file["id"], zip_name, LOCAL_FOLDER_PATH)

        print(f"Extracting {zip_name}...")
        unzip_and_clean(downloaded_zip_path, LOCAL_FOLDER_PATH)

        print(f"Finished processing {zip_name}.\n")

6. Running the main function

In [ ]:
if __name__ == "__main__":
    main()
